In [1]:
import pandas as pd
classified_reviews = pd.read_csv(r'C:\Users\vishw\OneDrive\Documents\classified_reviews_zeroshot.csv')

In [2]:
salary_data = pd.read_csv(r'C:\Users\vishw\OneDrive\Documents\DataMiningJobReview\salaries.csv')
salary_data.head()

,company_name,page,job_title,pay_range,min,max,per,num_salaries
0,HealthTrust Workforce Solutions,1,Registered Nurse,$76K - $95K /yr,76000,95000,yr,111
1,HealthTrust Workforce Solutions,1,RN,$78K - $101K /yr,78000,101000,yr,4
2,HealthTrust Workforce Solutions,1,Registered Nurse (RN),$74K - $94K /yr,74000,94000,yr,1
3,HealthTrust Workforce Solutions,1,"Registered Nurse, BSN",$79K - $100K /yr,79000,100000,yr,60
4,HealthTrust Workforce Solutions,1,"ER RN, BSN",$79K - $105K /yr,79000,105000,yr,1


In [5]:
demographics_data = pd.read_csv(r'C:\Users\vishw\OneDrive\Documents\DataMiningJobReview\demographics.csv')
demographics_data.head()

,company_name,category,group,rating,num_ratings
0,HealthTrust Workforce Solutions,Race / Ethnicity,★ Asian,2.5,17
1,HealthTrust Workforce Solutions,Race / Ethnicity,★ Black or African American,3.4,36
2,HealthTrust Workforce Solutions,Race / Ethnicity,★ Hispanic or Latinx,3.1,24
3,HealthTrust Workforce Solutions,Race / Ethnicity,★ White,3.3,54
4,HealthTrust Workforce Solutions,Gender,★ Men,3.1,38


In [7]:
"""
Pivot demographics from long format (one row per company/group) to wide
format (one row per company, groups as columns).
"""
import pandas as pd
import re

# Drop junk rows: Glassdoor's scroll-widget text sometimes gets scraped into
# the "group" field instead of a clean group name. These rows have rating=NaN
# and duplicate info already present in the properly-parsed rows below them.
demographics_data = demographics_data[demographics_data["rating"].notna()].copy()

def clean_group_name(g):
    g = g.replace("★", "").strip()          # drop the star
    g = g.lower()
    g = re.sub(r"[^a-z0-9]+", "_", g)        # spaces/slashes -> underscore
    g = g.strip("_")
    return g

demographics_data["group_clean"] = demographics_data["group"].apply(clean_group_name)

# Pivot ratings wide
ratings_wide = demographics_data.pivot_table(
    index="company_name",
    columns="group_clean",
    values="rating",
    aggfunc="first",
)
ratings_wide.columns = [f"{c}_rating" for c in ratings_wide.columns]

# Pivot ratings wide - this is the only output now (num_ratings dropped per request)
demographics_data_cleaned = ratings_wide.reset_index()


In [14]:
rating_cols = [c for c in demographics_data_cleaned.columns if c.endswith("_rating")]
 
# Track how many ratings were REAL before filling (lost once we overwrite NaNs)
demographics_data_cleaned["n_real_ratings"] = demographics_data_cleaned[rating_cols].notna().sum(axis=1)
 
# Fill every rating column at once, in place - each row's blanks get filled
# with that same row's own average across whichever ratings it does have.
demographics_data_cleaned[rating_cols] = demographics_data_cleaned[rating_cols].apply(lambda row: row.fillna(row.mean()), axis=1)
 

In [20]:
demographics_data_cleaned.head()

,company_name,asian_rating,black_or_african_american_rating,caregivers_rating,heterosexual_rating,hispanic_or_latinx_rating,indigenous_american_or_alaska_native_rating,lgbtq_rating,men_rating,middle_eastern_rating,non_veterans_rating,not_a_parent_or_caregiver_rating,parents_guardians_rating,people_with_disabilities_rating,people_without_disabilities_rating,transgender_and_or_non_binary_rating,veterans_rating,white_rating,women_rating,n_real_ratings
0,Avera Health,5.00,2.60,3.938462,3.8,4.70,3.938462,4.00,4.00,3.938462,4.0,3.7,4.30,3.60,3.9,3.938462,3.938462,3.90000,3.7,13
1,BJC HealthCare,3.70,3.40,3.600000,3.5,2.50,3.318750,3.30,4.00,3.318750,3.5,3.4,3.30,3.40,3.5,3.000000,2.500000,3.30000,3.2,16
2,CHRISTUS Health,3.60,3.90,2.900000,3.7,3.60,2.300000,3.10,4.00,3.525000,3.6,3.8,3.60,3.90,3.6,3.400000,3.800000,3.52500,3.6,16
3,Chicago Public Schools,3.50,3.40,3.200000,3.6,3.70,3.000000,3.10,3.60,3.343750,3.4,3.4,3.60,3.40,3.4,3.100000,2.700000,3.34375,3.4,16
4,College Achieve Public Schools,3.12,3.12,3.120000,3.4,3.12,3.120000,3.12,3.12,3.120000,3.1,2.9,3.12,3.12,3.3,3.120000,3.120000,3.12000,2.9,5


In [16]:
salary_data = pd.read_csv(r'C:\Users\vishw\OneDrive\Documents\DataMiningJobReview\salaries.csv')

In [18]:
salary_data.head(5)

,company_name,page,job_title,pay_range,min,max,per,num_salaries
0,HealthTrust Workforce Solutions,1,Registered Nurse,$76K - $95K /yr,76000,95000,yr,111
1,HealthTrust Workforce Solutions,1,RN,$78K - $101K /yr,78000,101000,yr,4
2,HealthTrust Workforce Solutions,1,Registered Nurse (RN),$74K - $94K /yr,74000,94000,yr,1
3,HealthTrust Workforce Solutions,1,"Registered Nurse, BSN",$79K - $100K /yr,79000,100000,yr,60
4,HealthTrust Workforce Solutions,1,"ER RN, BSN",$79K - $105K /yr,79000,105000,yr,1


In [22]:
salary_and_rating = pd.merge(salary_data, demographics_data_cleaned, left_on='company_name',right_on='company_name',how='left',suffixes=('_pay',''))

In [24]:
salary_and_rating.head()

,company_name,page,job_title,pay_range,min,max,per,num_salaries,asian_rating,black_or_african_american_rating,...,non_veterans_rating,not_a_parent_or_caregiver_rating,parents_guardians_rating,people_with_disabilities_rating,people_without_disabilities_rating,transgender_and_or_non_binary_rating,veterans_rating,white_rating,women_rating,n_real_ratings
0,HealthTrust Workforce Solutions,1,Registered Nurse,$76K - $95K /yr,76000,95000,yr,111,2.5,3.4,...,3.2,2.8,3.8,3.1,3.3,3.14,2.6,3.3,3.4,15
1,HealthTrust Workforce Solutions,1,RN,$78K - $101K /yr,78000,101000,yr,4,2.5,3.4,...,3.2,2.8,3.8,3.1,3.3,3.14,2.6,3.3,3.4,15
2,HealthTrust Workforce Solutions,1,Registered Nurse (RN),$74K - $94K /yr,74000,94000,yr,1,2.5,3.4,...,3.2,2.8,3.8,3.1,3.3,3.14,2.6,3.3,3.4,15
3,HealthTrust Workforce Solutions,1,"Registered Nurse, BSN",$79K - $100K /yr,79000,100000,yr,60,2.5,3.4,...,3.2,2.8,3.8,3.1,3.3,3.14,2.6,3.3,3.4,15
4,HealthTrust Workforce Solutions,1,"ER RN, BSN",$79K - $105K /yr,79000,105000,yr,1,2.5,3.4,...,3.2,2.8,3.8,3.1,3.3,3.14,2.6,3.3,3.4,15


In [46]:
salary_and_rating.columns

Index(['company_name', 'page', 'job_title', 'pay_range', 'min', 'max', 'per',
       'num_salaries', 'asian_rating', 'black_or_african_american_rating',
       'caregivers_rating', 'heterosexual_rating', 'hispanic_or_latinx_rating',
       'indigenous_american_or_alaska_native_rating', 'lgbtq_rating',
       'men_rating', 'middle_eastern_rating', 'non_veterans_rating',
       'not_a_parent_or_caregiver_rating', 'parents_guardians_rating',
       'people_with_disabilities_rating', 'people_without_disabilities_rating',
       'transgender_and_or_non_binary_rating', 'veterans_rating',
       'white_rating', 'women_rating', 'n_real_ratings'],
      dtype='str')

In [26]:
classified_reviews = pd.read_csv(r'C:\Users\vishw\OneDrive\Documents\classified_reviews_zeroshot.csv')

In [30]:
classified_reviews.head()

,company_name,department,occupation,page,rating,date,title,job_title,employment_status,location,...,overall_job_satisfaction_score,healthy_work-life_balance_score,pride_and_sense_of_purpose_score,compassion_fatigue_from_difficult_clients_or_students_score,frustration_with_management_or_leadership_score,burnout_and_exhaustion_score,high_staff_turnover_and_instability_score,dissatisfaction_with_pay_score,invisible_labor_score,positive_experience_score
0,HealthTrust Workforce Solutions,Healthcare,Nurse,1,4.0,"Jan 14, 2026",I love healthtrust,"Registered nurse, bsn",Current employee,NaN,...,0.989679,0.923319,0.746604,0.116621,0.041167,0.014943,0.006181,0.006165,-0.244137,2.759285
1,HealthTrust Workforce Solutions,Healthcare,Nurse,1,3.0,"Jan 3, 2026",Okay,"Registered nurse, bsn",Current employee,"San Antonio, TX",...,0.068793,0.148142,0.136294,0.004567,0.152505,0.001075,0.315970,0.520575,0.430878,0.341078
2,HealthTrust Workforce Solutions,Healthcare,Nurse,1,4.0,"Aug 20, 2025",Not a bad company to work for,"Registered nurse, bsn","Current employee, more than 3 years",Salt Lake City,...,0.640716,0.978437,0.099966,0.057352,0.534291,0.011752,0.233246,0.886876,0.831989,1.206703
3,HealthTrust Workforce Solutions,Healthcare,Nurse,1,4.0,"May 12, 2025",Peace at work,"Registered nurse, bsn","Current employee, more than 3 years","Richmond, VA",...,0.063112,0.116940,0.002046,0.015239,0.963927,0.201550,0.000127,0.997475,1.192720,0.123519
4,HealthTrust Workforce Solutions,Healthcare,Nurse,1,5.0,"Apr 10, 2025",Flexible per diem,"Registered nurse, bsn",Current employee,"Dallas, TX",...,0.027785,0.288605,0.019905,0.002574,0.071380,0.001633,0.448733,0.097827,0.302581,0.189539


In [44]:
classified_reviews.columns

Index(['company_name', 'department', 'occupation', 'page', 'rating', 'date',
       'title', 'job_title', 'employment_status', 'location', 'recommend',
       'ceo_approval', 'business_outlook', 'pros', 'cons', 'city', 'state',
       'region', 'full_text', 'supportive_coworkers_and_teamwork_score',
       'overall_job_satisfaction_score', 'healthy_work-life_balance_score',
       'pride_and_sense_of_purpose_score',
       'compassion_fatigue_from_difficult_clients_or_students_score',
       'frustration_with_management_or_leadership_score',
       'burnout_and_exhaustion_score',
       'high_staff_turnover_and_instability_score',
       'dissatisfaction_with_pay_score', 'invisible_labor_score',
       'positive_experience_score'],
      dtype='str')

In [48]:
classified_reviews["job_title_norm"] = classified_reviews["job_title"].str.strip().str.lower()
salary_and_rating["job_title_norm"] = salary_and_rating["job_title"].str.strip().str.lower()

final_data = pd.merge(
    classified_reviews,
    salary_and_rating.drop(columns=["page", "job_title"]),
    left_on=["company_name", "job_title_norm"],
    right_on=["company_name", "job_title_norm"],
    how="left",
    suffixes=('', '_Pr')
)

In [50]:
final_data.head()

,company_name,department,occupation,page,rating,date,title,job_title,employment_status,location,...,non_veterans_rating,not_a_parent_or_caregiver_rating,parents_guardians_rating,people_with_disabilities_rating,people_without_disabilities_rating,transgender_and_or_non_binary_rating,veterans_rating,white_rating,women_rating,n_real_ratings
0,HealthTrust Workforce Solutions,Healthcare,Nurse,1,4.0,"Jan 14, 2026",I love healthtrust,"Registered nurse, bsn",Current employee,NaN,...,3.2,2.8,3.8,3.1,3.3,3.14,2.6,3.3,3.4,15.0
1,HealthTrust Workforce Solutions,Healthcare,Nurse,1,3.0,"Jan 3, 2026",Okay,"Registered nurse, bsn",Current employee,"San Antonio, TX",...,3.2,2.8,3.8,3.1,3.3,3.14,2.6,3.3,3.4,15.0
2,HealthTrust Workforce Solutions,Healthcare,Nurse,1,4.0,"Aug 20, 2025",Not a bad company to work for,"Registered nurse, bsn","Current employee, more than 3 years",Salt Lake City,...,3.2,2.8,3.8,3.1,3.3,3.14,2.6,3.3,3.4,15.0
3,HealthTrust Workforce Solutions,Healthcare,Nurse,1,4.0,"May 12, 2025",Peace at work,"Registered nurse, bsn","Current employee, more than 3 years","Richmond, VA",...,3.2,2.8,3.8,3.1,3.3,3.14,2.6,3.3,3.4,15.0
4,HealthTrust Workforce Solutions,Healthcare,Nurse,1,5.0,"Apr 10, 2025",Flexible per diem,"Registered nurse, bsn",Current employee,"Dallas, TX",...,3.2,2.8,3.8,3.1,3.3,3.14,2.6,3.3,3.4,15.0


In [52]:
final_data.to_csv(r'C:\Users\vishw\OneDrive\Documents\final_data.csv', index=False)


In [68]:
import os
print(os.getcwd())

C:\Users\vishw\Documents\NLP_Learning
